In [1]:
import pandas as pd
import time
import os

In [2]:
import urllib.request
import gzip
import shutil

url = "https://snap.stanford.edu/data/sx-mathoverflow.txt.gz"
file_gz = "sx-mathoverflow.txt.gz"
file_txt = "sx-mathoverflow.txt"

# Download
if not os.path.exists(file_gz):
    print("Downloading dataset...")
    urllib.request.urlretrieve(url, file_gz)
    print("Download complete.")

# Unzip
if not os.path.exists(file_txt):
    print("Extracting dataset...")
    with gzip.open(file_gz, 'rb') as f_in:
        with open(file_txt, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
    print("Extraction complete.")

print("Dataset ready.")


Dataset ready.


In [3]:
# Convert to CSV format
df_raw = pd.read_csv("sx-mathoverflow.txt", sep=r"\s+", header=None, names=["SRC", "TGT", "Unix"])
df_raw.to_csv("sx-mathoverflow.csv", index=False)

print("Rows:", len(df_raw))
df_raw.head()


Rows: 506550


,SRC,TGT,Unix
0,1,4,1254192988
1,3,4,1254194656
2,1,2,1254202612
3,25,1,1254232804
4,14,16,1254263166


In [4]:
print("Number of rows:", len(df_raw))
print("Number of unique nodes:", len(set(df_raw["SRC"]).union(set(df_raw["TGT"]))))

df_raw.describe()


Number of rows: 506550
Number of unique nodes: 24818


,SRC,TGT,Unix
count,506550.000000,506550.000000,5.065500e+05
mean,12483.063415,15763.243717,1.347550e+09
std,15739.104958,18411.324079,5.872353e+07
min,1.000000,1.000000,1.254193e+09
25%,2051.000000,2811.000000,1.294387e+09
50%,6360.000000,8628.000000,1.343450e+09
75%,15630.000000,22002.000000,1.396941e+09
max,88580.000000,88580.000000,1.457262e+09


In [5]:
def clean_temporal_data(df):
    start = time.time()
    
    df = df.copy()
    
    # Cast types
    df["SRC"] = df["SRC"].astype(str)
    df["TGT"] = df["TGT"].astype(str)
    df["Unix"] = df["Unix"].astype(int)
    
    # Remove duplicates
    before = len(df)
    df = df.drop_duplicates(subset=["SRC", "TGT", "Unix"])
    duplicates_removed = before - len(df)
    
    # Sort by time
    df = df.sort_values(by=["Unix", "SRC", "TGT"]).reset_index(drop=True)
    
    # Add edge_id
    df.insert(0, "edge_id", ["e" + str(i).zfill(7) for i in range(1, len(df)+1)])
    
    end = time.time()
    
    print("Cleaning completed.")
    print("Rows after cleaning:", len(df))
    print("Duplicates removed:", duplicates_removed)
    print("Time taken (seconds):", round(end - start, 3))
    
    return df

df_clean = clean_temporal_data(df_raw)
df_clean.head()


Cleaning completed.
Rows after cleaning: 506523
Duplicates removed: 27
Time taken (seconds): 0.335


,edge_id,SRC,TGT,Unix
0,e0000001,1,4,1254192988
1,e0000002,3,4,1254194656
2,e0000003,1,2,1254202612
3,e0000004,3,1,1254206196
4,e0000005,1,1,1254207602


In [6]:
# edges_sorted structure
edges_sorted = list(zip(df_clean["Unix"], df_clean["SRC"], df_clean["TGT"], df_clean["edge_id"]))

print("First 5 edges_sorted:")
edges_sorted[:5]


First 5 edges_sorted:


[(1254192988, '1', '4', 'e0000001'),
 (1254194656, '3', '4', 'e0000002'),
 (1254202612, '1', '2', 'e0000003'),
 (1254206196, '3', '1', 'e0000004'),
 (1254207602, '1', '1', 'e0000005')]

In [7]:
adj_time = {}

for t, src, tgt, eid in edges_sorted:
    adj_time.setdefault(src, []).append((t, tgt, eid))

print("Example adjacency for first node:")
list(adj_time.items())[:1]


Example adjacency for first node:


[('1',
  [(1254192988, '4', 'e0000001'),
   (1254202612, '2', 'e0000003'),
   (1254207602, '1', 'e0000005'),
   (1254259818, '25', 'e0000008'),
   (1254271421, '16', 'e0000010'),
   (1254271943, '16', 'e0000011'),
   (1254272026, '2', 'e0000012'),
   (1254273043, '2', 'e0000014'),
   (1254273152, '22', 'e0000015'),
   (1254274939, '3', 'e0000019'),
   (1254276270, '2', 'e0000021'),
   (1254279260, '28', 'e0000022'),
   (1254302991, '27', 'e0000025'),
   (1254304365, '28', 'e0000026'),
   (1254392595, '7', 'e0000031'),
   (1254392913, '27', 'e0000032'),
   (1254395565, '32', 'e0000034'),
   (1254432016, '7', 'e0000038'),
   (1254432642, '32', 'e0000039'),
   (1254432924, '32', 'e0000040'),
   (1254441471, '37', 'e0000045'),
   (1254442478, '37', 'e0000046'),
   (1254444184, '2', 'e0000048'),
   (1254456232, '32', 'e0000052'),
   (1254456673, '37', 'e0000053'),
   (1254456896, '32', 'e0000054'),
   (1254459612, '37', 'e0000055'),
   (1254480209, '40', 'e0000059'),
   (1254481887, '42', '

In [8]:
print("FINAL SUMMARY")
print("---------------------")
print("Total cleaned edges:", len(df_clean))
print("Total unique nodes:", len(set(df_clean["SRC"]).union(set(df_clean["TGT"]))))
print("Memory usage (MB):", round(df_clean.memory_usage(deep=True).sum() / 1e6, 2))


FINAL SUMMARY
---------------------
Total cleaned edges: 506523
Total unique nodes: 24818
Memory usage (MB): 86.9


In [9]:
# Create smaller time window subset for demo

min_time = df_clean["Unix"].min()
max_time = df_clean["Unix"].max()

print("Time range:", min_time, "to", max_time)

# Example: take first 50,000 edges
df_subset = df_clean.iloc[:50000].copy()

print("Subset edges:", len(df_subset))
print("Subset memory (MB):", round(df_subset.memory_usage(deep=True).sum()/1e6, 2))


Time range: 1254192988 to 1457262355
Subset edges: 50000
Subset memory (MB): 8.5


In [10]:
df_clean.iloc[:50000]

,edge_id,SRC,TGT,Unix
0,e0000001,1,4,1254192988
1,e0000002,3,4,1254194656
2,e0000003,1,2,1254202612
3,e0000004,3,1,1254206196
4,e0000005,1,1,1254207602
...,...,...,...,...
49995,e0049996,3818,3032,1272195913
49996,e0049997,2734,2734,1272196360
49997,e0049998,2000,1946,1272196569
49998,e0049999,362,362,1272196902


In [11]:
# Time-window subset (example: 1 year window)

import datetime

# Convert Unix to datetime
df_clean["datetime"] = pd.to_datetime(df_clean["Unix"], unit="s")

start_date = df_clean["datetime"].min()
print("Start date:", start_date)

# Take first 365 days
end_date = start_date + pd.Timedelta(days=365)

df_time_window = df_clean[df_clean["datetime"] <= end_date].copy()

print("Edges in 1-year window:", len(df_time_window))
print("Memory (MB):", round(df_time_window.memory_usage(deep=True).sum()/1e6, 2))


Start date: 2009-09-29 02:56:28
Edges in 1-year window: 97221
Memory (MB): 18.11


In [12]:
# Auto-pick a time window to get ~50k edges (tunable)

import pandas as pd

df_tmp = df_clean.copy()
df_tmp["datetime"] = pd.to_datetime(df_tmp["Unix"], unit="s")

start_date = df_tmp["datetime"].min()
target_edges = 50000

# Search window sizes (days) to get close to target
candidates = [30, 60, 90, 120, 150, 180, 240, 300, 365]

best = None
for days in candidates:
    end_date = start_date + pd.Timedelta(days=days)
    df_win = df_tmp[df_tmp["datetime"] <= end_date]
    diff = abs(len(df_win) - target_edges)
    if best is None or diff < best[0]:
        best = (diff, days, len(df_win))

print("Best window:")
print("Days:", best[1], "Edges:", best[2], "Diff from target:", best[0])


Best window:
Days: 180 Edges: 42354 Diff from target: 7646


In [13]:
# Build final demo subset using the chosen days (replace DAYS with printed best[1])

DAYS = best[1]  # keep as found above
end_date = start_date + pd.Timedelta(days=DAYS)

df_demo = df_tmp[df_tmp["datetime"] <= end_date].copy()
df_demo = df_demo.drop(columns=["datetime"])

print("Demo edges:", len(df_demo))
print("Demo nodes:", len(set(df_demo["SRC"]).union(set(df_demo["TGT"]))))
print("Demo memory (MB):", round(df_demo.memory_usage(deep=True).sum()/1e6, 2))


Demo edges: 42354
Demo nodes: 2256
Demo memory (MB): 7.54
